<h1> Preparacion De Datos </h1> 

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv('/datasets/logs_exp_us.csv', sep='\t')

df.columns = ['event_name', 'user_id', 'timestamp', 'exp_id']

# Conversión de tipos
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df['date'] = df['timestamp'].dt.date

def check_hypothesis(successes1, successes2, trials1, trials2, alpha=0.05):
    p1 = successes1 / trials1
    p2 = successes2 / trials2
    p_combined = (successes1 + successes2) / (trials1 + trials2)
    difference = p1 - p2
    
    z_value = difference / math.sqrt(p_combined * (1 - p_combined) * (1/trials1 + 1/trials2))
    distr = stats.norm(0, 1)
    p_value = (1 - distr.cdf(abs(z_value))) * 2
    
    return p_value

In [4]:
events_sequence = ['MainScreenAppear', 'OffersScreenAppear', 'CartScreenAppear', 'PaymentScreenSuccessful']

for i in range(len(events_sequence)-1):
    current_users = df[df['event_name'] == events_sequence[i]]['user_id'].nunique()
    next_users = df[df['event_name'] == events_sequence[i+1]]['user_id'].nunique()
    conversion_rate = next_users / current_users * 100
    print(f"{events_sequence[i]} → {events_sequence[i+1]}: {conversion_rate:.1f}%")

MainScreenAppear → OffersScreenAppear: 62.0%
OffersScreenAppear → CartScreenAppear: 81.3%
CartScreenAppear → PaymentScreenSuccessful: 94.6%


In [6]:
def validate_aa_test(group_246, group_247, event_name):
    prop_246 = len(group_246[group_246['event_name'] == event_name]) / len(group_246)
    prop_247 = len(group_247[group_247['event_name'] == event_name]) / len(group_247)
    
    # Test de proporciones
    p_value = proportions_ztest([prop_246*len(group_246), prop_247*len(group_247)], 
                               [len(group_246), len(group_247)])[1]
    return p_value

In [7]:
num_tests = 5
alpha_original = 0.05
alpha_bonferroni = alpha_original / num_tests  # 0.01

print(f"Nivel de significancia ajustado: {alpha_bonferroni:.3f}")

Nivel de significancia ajustado: 0.010


In [8]:
daily_events = df.groupby('date')['event_name'].count()
complete_data_start = daily_events[daily_events > daily_events.quantile(0.95)].index[0]
print(f"Datos completos desde: {complete_data_start}")

Datos completos desde: 2019-08-01


In [9]:
import math
from statsmodels.stats.proportion import proportions_ztest

In [10]:
print("Grupos experimentales disponibles:")
print(df['exp_id'].value_counts())

df_clean = df[df['date'] >= pd.to_datetime('2019-08-01').date()]

Grupos experimentales disponibles:
248    85747
246    80304
247    78075
Name: exp_id, dtype: int64


In [11]:
group_246 = df_clean[df_clean['exp_id'] == 246]
group_247 = df_clean[df_clean['exp_id'] == 247]

# Test A/A para cada evento
events_to_test = ['MainScreenAppear', 'OffersScreenAppear', 'CartScreenAppear', 'PaymentScreenSuccessful']

print("=== TEST A/A (Validación de grupos de control) ===")
for event in events_to_test:
    p_value = validate_aa_test(group_246, group_247, event)
    print(f"{event}: p-value = {p_value:.4f}")

=== TEST A/A (Validación de grupos de control) ===
MainScreenAppear: p-value = 0.0000
OffersScreenAppear: p-value = 0.0000
CartScreenAppear: p-value = 0.0000
PaymentScreenSuccessful: p-value = 0.0000


In [12]:
group_248 = df_clean[df_clean['exp_id'] == 248]

print("\n=== TEST A/B (Control vs Experimental) ===")
for event in events_to_test:
    # Calcular proporciones
    prop_246 = len(group_246[group_246['event_name'] == event]) / len(group_246)
    prop_248 = len(group_248[group_248['event_name'] == event]) / len(group_248)
    
    # Test estadístico
    p_value = proportions_ztest([prop_246*len(group_246), prop_248*len(group_248)], 
                               [len(group_246), len(group_248)])[1]
    
    print(f"{event}: Control={prop_246:.4f}, Experimental={prop_248:.4f}, p-value={p_value:.4f}")


=== TEST A/B (Control vs Experimental) ===
MainScreenAppear: Control=0.4748, Experimental=0.4792, p-value=0.0726
OffersScreenAppear: Control=0.1860, Experimental=0.1935, p-value=0.0001
CartScreenAppear: Control=0.1852, Experimental=0.1794, p-value=0.0022
PaymentScreenSuccessful: Control=0.1500, Experimental=0.1435, p-value=0.0002


In [13]:
print("=== ANÁLISIS EXPLORATORIO ===")
print(f"Total de eventos: {len(df)}")
print(f"Total de usuarios únicos: {df['user_id'].nunique()}")
print(f"Promedio de eventos por usuario: {len(df) / df['user_id'].nunique():.2f}")

# Rango de fechas
print(f"Fecha mínima: {df['timestamp'].min()}")
print(f"Fecha máxima: {df['timestamp'].max()}")

# Frecuencia de eventos
print("\nFrecuencia de eventos:")
print(df['event_name'].value_counts())

=== ANÁLISIS EXPLORATORIO ===
Total de eventos: 244126
Total de usuarios únicos: 7551
Promedio de eventos por usuario: 32.33
Fecha mínima: 2019-07-25 04:43:36
Fecha máxima: 2019-08-07 21:15:17

Frecuencia de eventos:
MainScreenAppear           119205
OffersScreenAppear          46825
CartScreenAppear            42731
PaymentScreenSuccessful     34313
Tutorial                     1052
Name: event_name, dtype: int64


In [14]:
print("=== ANÁLISIS DE PÉRDIDA DE DATOS ===")
print(f"Eventos antes del filtro: {len(df)}")
print(f"Eventos después del filtro: {len(df_clean)}")
print(f"Eventos perdidos: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

print(f"Usuarios antes del filtro: {df['user_id'].nunique()}")
print(f"Usuarios después del filtro: {df_clean['user_id'].nunique()}")

=== ANÁLISIS DE PÉRDIDA DE DATOS ===
Eventos antes del filtro: 244126
Eventos después del filtro: 241298
Eventos perdidos: 2828 (1.2%)
Usuarios antes del filtro: 7551
Usuarios después del filtro: 7534


In [15]:
control_combined = pd.concat([group_246, group_247])

print("\n=== TEST A/B (Control Combinado vs Experimental) ===")
for event in events_to_test:
    # Control combinado
    prop_control = len(control_combined[control_combined['event_name'] == event]) / len(control_combined)
    # Experimental
    prop_248 = len(group_248[group_248['event_name'] == event]) / len(group_248)
    
    # Test estadístico
    p_value = proportions_ztest([prop_control*len(control_combined), prop_248*len(group_248)],
                               [len(control_combined), len(group_248)])[1]
    
    print(f"{event}: Control_Combinado={prop_control:.4f}, Experimental={prop_248:.4f}, p-value={p_value:.4f}")


=== TEST A/B (Control Combinado vs Experimental) ===
MainScreenAppear: Control_Combinado=0.4907, Experimental=0.4792, p-value=0.0000
OffersScreenAppear: Control_Combinado=0.1913, Experimental=0.1935, p-value=0.1927
CartScreenAppear: Control_Combinado=0.1735, Experimental=0.1794, p-value=0.0003
PaymentScreenSuccessful: Control_Combinado=0.1402, Experimental=0.1435, p-value=0.0259


In [16]:
print("\n=== ANÁLISIS COMPLETO DEL EMBUDO ===")

# Usuarios únicos por evento
for event in events_sequence:
    users_count = df_clean[df_clean['event_name'] == event]['user_id'].nunique()
    total_users = df_clean['user_id'].nunique()
    print(f"{event}: {users_count} usuarios ({users_count/total_users*100:.1f}%)")

# Porcentaje de usuarios que completan todo el journey
complete_journey = df_clean.groupby('user_id')['event_name'].apply(
    lambda x: all(event in x.values for event in events_sequence)
).sum()

print(f"\nUsuarios que completan todo el journey: {complete_journey} ({complete_journey/df_clean['user_id'].nunique()*100:.1f}%)")


=== ANÁLISIS COMPLETO DEL EMBUDO ===
MainScreenAppear: 7419 usuarios (98.5%)
OffersScreenAppear: 4593 usuarios (61.0%)
CartScreenAppear: 3734 usuarios (49.6%)
PaymentScreenSuccessful: 3539 usuarios (47.0%)

Usuarios que completan todo el journey: 3429 (45.5%)


In [ ]:
print("\n=== CONCLUSIONES DEL EXPERIMENTO ===")
print("Test A/A: Los grupos de control NO son equivalentes (p < 0.01)")
print("Esto indica problemas en la aleatorización del experimento")
print("\nTest A/B principales hallazgos:")
print("- OffersScreenAppear: Mejora significativa en grupo experimental")
print("- CartScreenAppear: Reducción significativa en grupo experimental") 
print("- PaymentScreenSuccessful: Reducción significativa en grupo experimental")

In [ ]:
print("\n=== RECOMENDACIONES ===")
print("1. NO implementar los cambios del grupo 248")
print("2. Investigar problemas de aleatorización")
print("3. Repetir el experimento con mejor diseño")
print("4. El nuevo diseño reduce las conversiones finales")